# LlmEdith : PyTorch sur 2×T4 (Kaggle)
*Accelerator* = **GPU T4 x2**, *Internet* = **ON**, secrets `HF_TOKEN` / `GITHUB_TOKEN`.
Sert aux ablations sur config `small`/`tiny` et aux **benchmarks** des checkpoints (en parallèle de la run TPU).
Les T4 n'ont pas de bf16 : l'entraînement passe automatiquement en fp16 + GradScaler.

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ['HF_TOKEN'] = s.get_secret('HF_TOKEN')
try:
    GH = s.get_secret('GITHUB_TOKEN')  # seulement si le dépôt GitHub est privé
except Exception:
    GH = None
os.environ['LLMEDITH_HUB_REPO'] = 'TON_PSEUDO_HF/llmedith-checkpoints'
repo = 'https://github.com/HarpeLm/LlmEdith.git'.replace('https://', f'https://{GH}@') if GH else 'https://github.com/HarpeLm/LlmEdith.git'
if not os.path.exists('LlmEdith'):
    subprocess.run(['git', 'clone', '-q', repo, 'LlmEdith'], check=True)
else:
    subprocess.run(['git', '-C', 'LlmEdith', 'pull', '-q'], check=True)
os.chdir('LlmEdith')

In [ ]:
!pip install -q -e '.[eval]'

## A. Ablation / pré-entraînement multi-GPU

In [ ]:
!torchrun --nproc_per_node 2 -m llmedith.torch_impl.train --config configs/small.yaml --resume \
    --max_minutes 500 --override data.hub_repo=TON_PSEUDO_HF/llmedith-data train.micro_batch_seqs=8

## B. Benchmarks du dernier checkpoint de la run principale

In [ ]:
from llmedith import hub
hub.pull_folder('TON_PSEUDO_HF/llmedith-checkpoints', 'main/latest', 'runs')
!python -m convert.to_hf --ckpt runs/main/latest --out exports/llmedith-main
!python -m eval.run_benchmarks --models edith-main=exports/llmedith-main --baselines --force